# Working with operators
`qse` provides an `Operator` class and an `Operators` class to easily manipulate quantum operators. The `Operators` class can be easily created from a `Qbits` object, making creating interaction hamiltonians very convienient.

In [ ]:
import qse

## The Operator class
The `Operator` class represents a single quantum operator.

In [ ]:
# To create 0.3 ZII
qse.Operator(operator="Z", qubits=0, nqbits=3, coef=0.3)

In [ ]:
# To create -2. IIXIY
qse.Operator(operator=["X", "Y"], qubits=[2, 4], nqbits=5, coef=-2)

Operators can be multiplied by a float, which multiplies the coefficient.

In [ ]:
op = qse.Operator("Z", 0, 3)
print(op)
print(op * 4.4)

An operator can be converted into a QuTiP object by the `to_qutip` method.

In [ ]:
# Create ZII
# Note that `coef` defaults to 1
op = qse.Operator(operator="Z", qubits=0, nqbits=3)
op.to_qutip()

# The Operators class
For dealing with multiple `Operator` classes we have the `Operators` class.

We can fill the `Operators` class either by passing `Operator` classes as a list or by addition.

In [ ]:
# Creating Operators by passing a list
nqbits = 3
qse.Operators(
    [qse.Operator(operator="Z", qubits=i, nqbits=nqbits) for i in range(nqbits)]
)

In [ ]:
# Creating Operators by addition
hamiltonian = qse.Operators()

for i in range(nqbits):
    hamiltonian += qse.Operator(operator="Z", qubits=i, nqbits=nqbits)

hamiltonian

We can use both methods to create complex models, for example to create a Heisenberg Model
$$ H = -J \sum_{i=1}^{N-1} (X_i X_{i+1} + Y_i Y_{i+1} + Z_i Z_{i+1})$$

In [ ]:
# For example
nqbits = 3
coupling = -1.0
ham_xx = qse.Operators(
    [qse.Operator("X", [i, i + 1], nqbits, coupling) for i in range(nqbits - 1)]
)
ham_yy = qse.Operators(
    [qse.Operator("Y", [i, i + 1], nqbits, coupling) for i in range(nqbits - 1)]
)
ham_zz = qse.Operators(
    [qse.Operator("Z", [i, i + 1], nqbits, coupling) for i in range(nqbits - 1)]
)
ham = ham_xx + ham_yy + ham_zz
ham

The `Operators` class also supports conversion to `QuTiP`

In [ ]:
ham.to_qutip()

## Creating an interaction Hamiltonian from a Qbits object
`Qbits` objects have a method `compute_interaction_hamiltonian` which allows one to easily create an interaction Hamiltonian from a `Qbits` geometry.

In [ ]:
# Create a nearest-neighbor Ising model on a hexagonal lattice
qbits = qse.lattices.hexagonal(1.0, 2, 3)
del qbits[[0, 11]]
qbits.labels = range(qbits.nqbits)  # relabel
qbits.draw(show_labels=True, radius="nearest")

In [ ]:
coupling = 1.0


# You can either explicity create a callable to pass to the function
def distance_func(d):
    return coupling * (d < 1.01)  # will be non-zero for nearest-neighbours only


ham_int = qbits.compute_interaction_hamiltonian(distance_func, interaction="Z")

# Or use a lambda function
# ham_int = qbits.compute_interaction_hamiltonian(lambda d: coupling * (d < 1.01), interaction="Z")

ham_int

In [ ]:
# Now add an external field to create the full hamiltonian
coupling_ext = 0.1
ham_ext = qse.Operators(
    [qse.Operator("X", i, qbits.nqbits, coupling_ext) for i in range(qbits.nqbits)]
)
ham_ext

In [ ]:
ham = ham_int + ham_ext
ham

## Version

In [ ]:
qse.utils.print_environment()